## Calculating the Units for homes withing SCE

This notebook utilizes parcel data, Zillow data, building data, and a SCE shapefile to calculate missing multi-family unit data. Residential parcels are subset to calculate building volume and create a regression that gives best approximations of multi-family home data. Single family home volume is additionally calculated. This unit data is utilized in calculating hosting capacity to assess where limited distributed energy resources exist across California.



In [2]:
# import necessary libraries
import pandas as pd
from shapely.geometry import box
import numpy as np
import geopandas as gpd
import os
import matplotlib.pyplot as plt
import fiona
import pandas as pd

# statistical libraries
#import sys
#!{sys.executable} -m pip install statsmodels
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

In [3]:
# set option to see all data frame columns
pd.set_option('display.max_columns', None)

In [4]:
# load parcels 
parcels = gpd.read_parquet("/../../capstone/electrigrid/data/parcels/parcels_final.parquet").to_crs(epsg=4326)

In [5]:
# import Zillow data 
zillow = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/zillow.parquet").to_crs(epsg=4326)

Building Footprint

Access: https://sat-io.earthengine.app/view/gba

In [6]:
# import building footprint as geopandas dataframe (may take 1-5 minutes)
building = gpd.read_parquet("../../../../../capstone/electrigrid/data/buildings/buildings_ca.parquet").to_crs(epsg=4326)

In [7]:
# load in SDG&E boundary map
sce_shape = gpd.read_file("/../../capstone/electrigrid/data/utilities/IOU_shapefiles.geojson")

sce_shape = sce_shape[sce_shape['Acronym'] == "SCE"]

## Streamlined unit-regression for multi-family homes and volume calculations for all homes

Takes approximately 

In [8]:
# only keep the geometry and parcel number in the parcel dataframe
parcels = parcels[['PARNO', 'geometry']]

## select only multi-family data from Zillow
zillow_multi = zillow[zillow['type'] == "Multi"]
zillow_multi = zillow_multi[zillow_multi['code'] != "RR106"]

# keep only multi family homes within pge
zillow_multi = zillow_multi.clip(sce_shape)

## crop only to residential parcels
# keep the indices where multi-family homes match to parcels
valid_parcels = parcels.sjoin(zillow_multi, predicate="contains").index.unique()

# select the parcels that match these indices
parcels_res = parcels.loc[valid_parcels]

# confirm that joining with Zillow decreases the number of parcels
assert len(parcels_res) < len(parcels)

# crop to residential buildings (by keeping only those within residential parcels)
valid_buildings = building.sjoin(parcels_res, predicate="intersects").index.unique()
buildings_res = building.loc[valid_buildings]

## join parcels to buildings (keeping observations as parcels, but with building attributes)
# sum number of units per parcel
units_sum = parcels_res.sjoin(zillow_multi, predicate="intersects").groupby(level=0)["unit"].sum()

# join on parcels with summed number of units
parcels_res = parcels_res.join(units_sum)

# confirm that joining with Zillow decreased the number of buildings
assert len(buildings_res) < len(building)

# keep all residential buildings, and add zillow points only where they match up
building_zillow = gpd.sjoin(
    buildings_res,
    zillow_multi,
    predicate = "intersects")

# drop the unnecessary index
building_zillow = building_zillow.drop(columns = "index_right")

# add in the parcel number to add all of the building volumes for the regression
building_zillow_parcels = building_zillow.sjoin(
                                parcels_res,
                                how = "left",
                                predicate = "intersects")

## FOR BUILDINGS INTERSECTING MORE THAN ONE PARCEL CALCULATE WHICH PARCEL IT INTERSECTS MORE
# change the crs to a projected crs
building_zillow_parcels = building_zillow_parcels.to_crs("EPSG:6933")
parcels_res = parcels_res.to_crs("EPSG:6933")

In [12]:
print(f"There are ",building_zillow_parcels['PARNO'].isna().sum(), "buildings not associated with a parcel.")

# check the number of zillow points that don't have parcels associated with them but have units
print(f"Of all of the homes that are not within parcels", building_zillow_parcels[building_zillow_parcels['PARNO'].isna()]['unit_left'].isna().sum(), " also do not have units.")

There are  11 buildings not associated with a parcel.
Of all of the homes that are not within parcels 8  also do not have units.


We could approximate units based on their code. 

In [13]:
building_zillow_parcels[building_zillow_parcels['PARNO'].isna()]

,source,id,height,var,region,bbox,geometry,type,year,room,heat,cool,own,unit_left,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,index_right,PARNO,unit_right
740308,osm,1002813197,3.831952,0.428591,USA,"{'xmax': -118.537831, 'xmin': -118.53801399999...","POLYGON ((-11437289.627 4439014.549, -11437274...",Multi,1993.0,3.0,None,None,O,NaN,396916.0,living,2787.0,1447086,06027000200,1192,PGE/SCE,RI101,NaN,NaN,NaN
743931,osm,787321635,4.389750,3.483013,USA,"{'xmax': -118.40144489999997, 'xmin': -118.401...","POLYGON ((-11424126.004 4443859.271, -11424114...",Multi,1950.0,5.0,None,None,O,NaN,637500.0,living,2434.0,1444106,06027000400,1192,PGE/SCE,RI102,NaN,NaN,NaN
744416,osm,787322206,6.420725,2.746906,USA,"{'xmax': -118.39218959999998, 'xmin': -118.392...","POLYGON ((-11423235.021 4443449.119, -11423222...",Multi,1958.0,NaN,None,None,None,NaN,311459.0,living,3612.0,1442573,06027000400,1192,PGE/SCE,RI104,NaN,NaN,NaN
744534,ms,UnitedStates_023010331_3674,4.307594,0.429483,USA,"{'xmax': -118.39265884266203, 'xmin': -118.392...","POLYGON ((-11423267.356 4443193.736, -11423267...",Multi,1919.0,4.0,None,None,O,NaN,226008.0,living,2564.0,1442706,06027000400,1192,PGE/SCE,RI101,NaN,NaN,NaN
744585,osm,604519969,3.068873,0.596099,USA,"{'xmax': -118.3977807, 'xmin': -118.3979026999...","POLYGON ((-11423773.221 4443338.798, -11423773...",Multi,1949.0,2.0,fossil,central,None,2.0,288680.0,living,1599.0,1442556,06027000400,1192,PGE/SCE,RI101,NaN,NaN,NaN
744602,ms,UnitedStates_023010331_8124,3.697568,1.527954,USA,"{'xmax': -118.39739396414652, 'xmin': -118.397...","POLYGON ((-11423724.228 4443241.150, -11423735...",Multi,1930.0,4.0,None,None,None,NaN,388620.0,living,1433.0,1442661,06027000400,1192,PGE/SCE,RI101,NaN,NaN,NaN
744607,osm,761326202,4.630708,1.861560,USA,"{'xmax': -118.39680000000001, 'xmin': -118.396...","POLYGON ((-11423676.860 4443221.567, -11423682...",Multi,1905.0,3.0,fossil,central,I,NaN,561000.0,living,1574.0,1442671,06027000400,1192,PGE/SCE,RI102,NaN,NaN,NaN
747129,osm,827035865,4.866893,1.128860,USA,"{'xmax': -118.4301005, 'xmin': -118.4302823, '...","POLYGON ((-11426897.408 4441496.390, -11426887...",Multi,1940.0,3.0,fossil,central,None,NaN,632596.0,living,2547.0,1446576,06027000300,1192,PGE/SCE,RI101,NaN,NaN,NaN
1238220,osm,1179246888,3.097831,1.187995,USA,"{'xmax': -119.01057949999999, 'xmin': -119.010...","POLYGON ((-11482906.401 4308642.471, -11482906...",Multi,1990.0,4.0,fossil,central,None,2.0,111812.0,living,1452.0,10637104,06107004104,1284,PGE/SCE,RI101,NaN,NaN,NaN
1238225,osm,1179246891,3.226012,0.693167,USA,"{'xmax': -119.01059379999997, 'xmin': -119.010...","POLYGON ((-11482890.133 4308605.179, -11482889...",Multi,1990.0,6.0,fossil,central,None,2.0,163031.0,living,1992.0,10637103,06107004104,1284,PGE/SCE,RI101,NaN,NaN,NaN


In [14]:
# drop rows with no parcel match before computing intersection
building_zillow_parcels = building_zillow_parcels.dropna(subset=['index_right'])
building_zillow_parcels['index_right'] = building_zillow_parcels['index_right'].astype(int)

# calculate intersection area for each building-parcel pair
building_zillow_parcels["intersection_area"] = (
    building_zillow_parcels.geometry.values
    .intersection(parcels_res.loc[building_zillow_parcels["index_right"]].geometry.values).area)

# keep only the parcel with the largest overlap per building
building_zillow_parcels = (
    building_zillow_parcels
    .sort_values("intersection_area", ascending=False)
    .loc[~building_zillow_parcels.index.duplicated(keep="first")]
    .drop(columns="intersection_area"))

# drop the unnecessary columns
building_zillow_parcels = building_zillow_parcels.drop(columns = ['index_right', 'unit_left'])
building_zillow_parcels = building_zillow_parcels.rename(columns = {'unit_right': 'total_unit'})

# reproject data frame to crs with meters as units
building_m = building_zillow_parcels.to_crs("EPSG:6933")

# create column from polygon area
building_m['area_m2'] = building_m.geometry.area

# rename height column to be clear about units
building_m.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
building_m['volume_m3'] = building_m['area_m2'] * building_m['height_m']

# keep only observations with unit data
building_w_units = building_m[~building_m['total_unit'].isna()]

assert building_w_units['total_unit'].isna().sum() == 0

# drop multi-family homes where the total_unit is equal to zero
building_w_units = building_w_units.drop(building_w_units[building_w_units['total_unit'] == 0].index)

# aggregate the volumes by parcel
total_volume_m3 = building_w_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
total_volume_m3 = pd.DataFrame(total_volume_m3)

# rename the column to total_volume_m3
total_volume_m3 = total_volume_m3.rename(columns = {'volume_m3' : 'total_volume_m3'})

# add the total volume to the buildings dataframe
building_w_units = building_w_units.join(total_volume_m3, on = 'PARNO')

# some of the homes don't have a parcel number (so replace the total volume with volume if its na
building_w_units['total_volume_m3'] = building_w_units['total_volume_m3'].fillna(building_w_units['volume_m3'])

# remove duplicates from the parcel number so that it doesn't skew the linear regression
building_w_units = building_w_units.drop_duplicates(subset = 'PARNO', keep = 'first')

# run model
results = smf.ols('total_unit ~ total_volume_m3', data=building_w_units).fit()

# add residuals as a column
building_w_units['residual'] = results.resid.copy()

# keep only observations that are less/equal to 2 standard deviations from residuals
building_units_clean = building_w_units[building_w_units['residual'].abs() <= 2 * building_w_units['residual'].std()]

# save outliers, as we will re-predict them using the regression
building_outliers = building_w_units[building_w_units['residual'].abs() > 2 * building_w_units['residual'].std()]

# rerun linear regression
results_clean = smf.ols('total_unit ~ total_volume_m3', data=building_units_clean).fit()

# save variables
intercept = results_clean.params[0]
slope = results_clean.params[1]

# extract just the multi-family homes where unit info is missing
missing_units = building_m[(building_m['total_unit'].isna()) & (building_m['total_unit'] == 0)]


# aggregate the volumes for the unit regression by parcel
missing_total_volume_m3 = missing_units.groupby('PARNO')['volume_m3'].sum(min_count = 1)

# change the series to a dataframe
missing_total_volume_m3 = pd.DataFrame(missing_total_volume_m3)

# rename the column to missing_total_volume_m3
missing_total_volume_m3 = missing_total_volume_m3.rename(columns = {'volume_m3' : 'missing_total_volume_m3'})

# add the missing total volume to the buildings dataframe
missing_units = missing_units.join(missing_total_volume_m3, on = 'PARNO')

# combine dataframes with missing unit data as well as outliers (since both will be predicted)
missing_outlier_units = pd.concat([building_outliers, missing_units])

assert len(missing_units) < len(missing_outlier_units)

# replace anywhere the missing_total_volume is missing (fill in for the outliers since total_volume was computed before)
missing_outlier_units['missing_total_volume_m3'] = missing_outlier_units['missing_total_volume_m3'].fillna(missing_outlier_units['total_volume_m3'])

# replace unit column with prediction
missing_outlier_units_pred = missing_outlier_units.copy().drop('total_unit', axis = 1)

missing_outlier_units_pred = missing_outlier_units_pred.reset_index(drop=True)

missing_outlier_units_pred['total_unit'] = round(intercept + missing_outlier_units_pred['missing_total_volume_m3'] * slope)

## multiple buildings in one parcel have duplicated units since they were computed on total volume
#  only keep the first observation per parcel
missing_outlier_units_pred = missing_outlier_units_pred.drop_duplicates(subset = 'PARNO', keep = 'first')

# combine multi-family homes data frames
multi_complete = pd.concat([building_units_clean, missing_outlier_units_pred]).to_crs(zillow.crs)

# fill the total_volume for those with the missing_total_volume_m3 and rename to total_volume_m3
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# fill the total_volume for those with the missing_total_volume_m3 
multi_complete['missing_total_volume_m3'] = multi_complete['missing_total_volume_m3'].fillna(multi_complete['total_volume_m3'])

# drop unnecessary columns 
multi_complete = multi_complete.drop(['residual', 'geometry', 'total_volume_m3'], axis = 1)

# rename to total_volume_m3
multi_complete = multi_complete.rename(columns = {'missing_total_volume_m3' : 'total_volume_m3'})

# drop the unit data from parcel res as the new unit data was computed
parcels_res = parcels_res.drop(columns = ['unit'])

# join the parcel data on PARNO to get the parcel geometry
multi_by_parcel = parcels_res.merge(multi_complete, on = 'PARNO')

## IN ORDER TO CONCAT THE MULTI-FAMILY HOMES WITH SINGLE FAMILY HOMES THEY MUST ALL BE POINTS
# set the geometry to the parcel geometry
multi_by_parcel = multi_by_parcel.set_geometry('geometry')

# change the crs 
multi_by_parcel = multi_by_parcel.to_crs("EPSG:6933")

# create centroids for each multi residential parcel 
multi_by_parcel['centroids'] = multi_by_parcel.geometry.centroid

# drop the geometry of multi_by_parcel so the centroid becomes the geometry
multi_by_parcel = multi_by_parcel.drop(columns = 'geometry')

# rename the centroid to geometry
multi_by_parcel = multi_by_parcel.rename(columns = {'centroids' : 'geometry'})

# set the centroid to the geometry
multi_by_parcel_processed = multi_by_parcel.set_geometry('geometry')

# change the CRS back to the Zillow CRS 
multi_by_parcel_processed = multi_by_parcel_processed.to_crs(zillow.crs)

## SET UNITS FOR SINGLE FAMILY HOMES 
# subset for single family zillow and condos
single_zillow = zillow[zillow['type'] == "Single"].to_crs(zillow.crs)

# keep only single family homes within pge
single_zillow = single_zillow.clip(sce_shape)

# change all single_zillow to unit = 1
single_zillow['total_unit'] = 1

# drop the unit column
single_zillow = single_zillow.drop(['unit'], axis = 1)

## CALCULATE AREA FOR SINGLE FAMILY HOME
# select parcels where single family homes exist 
single_parcels = parcels.sjoin(single_zillow, predicate="contains").index.unique()
single_parcels = parcels.loc[single_parcels]

# crop to residential buildings (by keeping only those within single residential parcels)
single_buildings = building.sjoin(single_parcels, predicate="intersects").index.unique()
buildings_res = building.loc[single_buildings]

# keep all single-family buildings, and add zillow points only where they match up
single_building_zillow = gpd.sjoin(
    buildings_res,
    single_zillow,
    predicate = "intersects")

# drop unnecessary columns
single_building_zillow = single_building_zillow.drop(columns = "index_right")

# reproject data frame to crs with meters as units
single_building_zillow = single_building_zillow.to_crs("EPSG:6933")

# create column from polygon area
single_building_zillow['area_m2'] = single_building_zillow.area

# rename height column to be clear about units
single_building_zillow.rename(columns={"height":"height_m"}, inplace = True)

# create volume column
single_building_zillow['volume_m3'] = single_building_zillow['area_m2'] * single_building_zillow['height_m']

# change the crs back to projected
single_building_zillow = single_building_zillow.to_crs(epsg=4326)

assert single_building_zillow.crs == multi_by_parcel_processed.crs

## FINAL CONCAT OF THE MULTI AND SINGLE FAMILY HOMES
# concat the single and multifamily dataframes
complete_zillow = pd.concat([multi_by_parcel_processed, single_building_zillow], axis = 0)

/tmp/ipykernel_2992145/3742501052.py:75: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  intercept = results_clean.params[0]
/tmp/ipykernel_2992145/3742501052.py:76: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  slope = results_clean.params[1]


# Final results 

In [15]:
complete_zillow.shape

(2279957, 26)

In [17]:
print(f"The number of homes calculated in the sce area", complete_zillow['total_unit'].sum())

The number of homes calculated in the sce area 2593343.0


## Renaming and saving

In [18]:
# save the complete Zillow points
complete_zillow_sce = complete_zillow.copy()

In [ ]:
# takes a really long time 
# complete_zillow_sce.to_parquet("complete_zillow_sce.parquet")

In [ ]:
sce_homes = gpd.read_parquet("/../../capstone/electrigrid/data/buildings/final_zillow_w_volume/complete_zillow_sce.parquet")

In [ ]:
sce_homes.head()

,PARNO,source,id,height_m,var,region,bbox,type,year,room,heat,cool,own,value,sqft_type,sqft,ID,GEOID,p_ID,area,code,total_unit,area_m2,volume_m3,total_volume_m3,geometry
0,48-6331-3-2,osm,470801986,7.360551,3.760925,USA,"{'xmax': -122.11905469999998, 'xmin': -122.119...",Multi,1970.0,3.0,None,None,O,1447455.0,living,2621.0,135866,06001409900,262,PGE/SCE,RI101,2.0,344.514287,2535.814937,2535.814937,POINT (-122.11915 37.75188)
1,45-5419-37,osm,467433407,3.687969,0.562078,USA,"{'xmax': -122.1772724, 'xmin': -122.1774869999...",Multi,1965.0,6.0,None,None,I,518707.0,living,3038.0,113661,06001409200,568,PGE/SCE,RI103,4.0,180.790440,666.749620,1476.620600,POINT (-122.17738 37.72601)
2,48-6231-27-1,osm,471260067,12.743473,1.726028,USA,"{'xmax': -122.12957120000002, 'xmin': -122.129...",Multi,1955.0,7.0,None,None,O,1335766.0,living,4473.0,135372,06001410000,262,PGE/SCE,RI101,2.0,452.441520,5765.676322,5765.676322,POINT (-122.12977 37.75103)
3,5-406-26,osm,473407799,5.020492,0.822397,USA,"{'xmax': -122.2846471, 'xmin': -122.2848646, '...",Multi,1905.0,4.0,None,None,None,270454.0,living,2625.0,162813,06001402400,468,PGE/SCE,RI101,2.0,165.396340,830.370936,1936.266938,POINT (-122.28473 37.81260)
4,10-774-20,osm,310133683,4.070246,1.512848,USA,"{'xmax': -122.25537620000001, 'xmin': -122.255...",Multi,1912.0,10.0,None,None,None,845491.0,living,8653.0,3077,06001403600,511,PGE/SCE,RI104,6.0,390.481853,1589.357102,1589.357102,POINT (-122.25553 37.81102)
